# FEM assembly and reordering

A finite element matrix is a sum of small ones. Each element contributes a
local matrix that couples its own nodes, and assembly adds that into the global
matrix at those nodes' rows and columns. Where elements share a node, their
contributions add up. The node numbering comes from the mesh generator. It
decides where the nonzeros fall, but not what the matrix means, and a better
numbering can be chosen afterwards.

This notebook assembles the C++ example's 1-D mesh, then a real 2-D one: P1
triangles on the unit square, with the nodes numbered at random, as an
unstructured mesh generator might. Then it reorders that matrix with reverse
Cuthill–McKee (RCM), and measures what reordering changes and what it leaves
alone.

**Uses:** `mtl5.sparse.coo_matrix` and `.tocsr`, `mtl5.sparse.rcm`,
`mtl5.sparse.cholesky` with `ordering=`, `mtl5.generators.laplacian_1d`,
`mtl5.generators.laplacian_2d`, `mtl5.eigvalsh`.

**From C++:** [`phase11_assembly_reorder/fem_assembly_reorder.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase11_assembly_reorder/fem_assembly_reorder.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
from matplotlib.colors import ListedColormap

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)


def bandwidth(S):
    # max |i - j| over the stored entries
    S = sp.coo_matrix(S)
    return int(np.abs(S.row - S.col).max())


def profile(S):
    # Sum over rows of the distance from the first stored entry to the diagonal:
    # the entries a banded (envelope) factorization stores below the diagonal.
    S = sp.csr_matrix(S)
    return int(sum(i - S.indices[S.indptr[i] : S.indptr[i + 1]].min() for i in range(S.shape[0])))


def spy(ax, S, title, color=SERIES[0], pixels=300):
    # The stored entries (explicit zeros included) on a grid of at most
    # pixels x pixels cells; a cell is colored if it holds any entry. Drawing one
    # cell per matrix entry would drop entries when the image is scaled down.
    S = sp.coo_matrix(S)
    bins = min(S.shape[0], pixels)
    counts, _, _ = np.histogram2d(S.row, S.col, bins=bins, range=[[0, S.shape[0]], [0, S.shape[1]]])
    extent = (-0.5, S.shape[1] - 0.5, S.shape[0] - 0.5, -0.5)  # in matrix coordinates
    ax.imshow(
        counts > 0, cmap=ListedColormap([SURFACE, color]), interpolation="none", extent=extent
    )
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

## Assembly in 1-D: duplicates add up

The C++ example's mesh is 4 elements on 5 nodes. Element e joins nodes e and
e + 1, and its stiffness matrix is [[1, −1], [−1, 1]]. In C++,
`shifted_inserter` with `update_plus` places each local matrix at an offset and
adds where entries overlap. In Python the same assembly is a list of triplets:
each element appends its four, with global indices, and COO sums repeated
coordinates when it compresses to CSR.

In [ ]:
n_elem = 4
k_local = np.array([[1.0, -1.0], [-1.0, 1.0]])
rows, cols, vals = [], [], []
for e in range(n_elem):
    nodes = np.array([e, e + 1])
    rows += list(np.repeat(nodes, 2))
    cols += list(np.tile(nodes, 2))
    vals += list(k_local.ravel())

K_coo = ms.coo_matrix(np.array(rows), np.array(cols), np.array(vals), (n_elem + 1, n_elem + 1))
K = ms.to_scipy(K_coo.tocsr()).toarray()
print(f"{K_coo.nnz} triplets -> {np.count_nonzero(K)} entries")
print(K)

# Interior nodes belong to two elements, so their diagonal sums two contributions.
assert K_coo.nnz == 4 * n_elem
assert np.array_equal(np.diag(K), [1, 2, 2, 2, 1])
# Away from the two end nodes this is the [-1, 2, -1] Laplacian.
L1 = ms.to_scipy(mtl5.generators.laplacian_1d(n_elem + 1)).toarray()
assert np.array_equal(K[1:-1, 1:-1], L1[1:-1, 1:-1])
assert np.allclose(K @ np.ones(n_elem + 1), 0)  # constants are free: no boundary condition yet

## A 2-D mesh, numbered at random

The C++ example's 2-D part uses a made-up 3 × 3 local matrix. Here the local
matrices are real: P1 (linear) triangles for −Δu = f on the unit square, an
m × m grid of cells, each cut into two triangles. For a triangle with vertices
(xᵢ, yᵢ), the local stiffness matrix is (bbᵀ + ccᵀ) / (4 × area), with
bᵢ = yⱼ − yₖ and cᵢ = xₖ − xⱼ over the cyclic triples (i, j, k).

The nodes get a random permutation of their grid labels, standing in for a
mesh generator's arbitrary numbering. Assembly doesn't care: each element
appends 9 triplets at its nodes' labels.

In [ ]:
m = 24
nx = m + 1
xy = np.array([(i / m, j / m) for j in range(nx) for i in range(nx)])
triangles = []
for j in range(m):
    for i in range(m):
        a = j * nx + i
        b, c, d = a + 1, a + nx, a + nx + 1
        triangles += [(a, b, d), (a, d, c)]
triangles = np.array(triangles)


def p1_stiffness(p):
    x, y = p[:, 0], p[:, 1]
    b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
    c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
    area = 0.5 * abs((x[1] - x[0]) * (y[2] - y[0]) - (x[2] - x[0]) * (y[1] - y[0]))
    return (np.outer(b, b) + np.outer(c, c)) / (4 * area)


label = np.random.default_rng(147).permutation(nx * nx)  # node k is called label[k]
rows, cols, vals = [], [], []
for tri in triangles:
    g = label[tri]
    rows.append(np.repeat(g, 3))
    cols.append(np.tile(g, 3))
    vals.append(p1_stiffness(xy[tri]).ravel())
K_coo = ms.coo_matrix(
    np.concatenate(rows), np.concatenate(cols), np.concatenate(vals), (nx * nx, nx * nx)
)
K = ms.to_scipy(K_coo.tocsr()).tocsr()
print(f"{len(triangles)} triangles, {nx * nx} nodes:", end=" ")
print(f"{K_coo.nnz:,} triplets -> {K.nnz:,} stored entries")

assert K_coo.nnz == 9 * len(triangles)
assert abs(K - K.T).max() == 0  # symmetric
assert np.abs(K @ np.ones(nx * nx)).max() < 1e-14  # constants in the null space

On this mesh the P1 matrix is an old friend. Put the nodes back in grid order
and keep the interior ones, which is the Dirichlet condition u = 0 on the
boundary, and the result is the 5-point Laplacian of
`mtl5.generators.laplacian_2d`. The coupling across each triangle's diagonal
edge comes out exactly zero, because that edge's two triangles cancel. The
entry is still stored, though: assembly decides the pattern from the elements,
before any values are added up.

In [ ]:
grid_order = K[label][:, label]  # row k is node k again
interior = np.array([j * nx + i for j in range(1, m) for i in range(1, m)])
K_int = grid_order[interior][:, interior]
L2 = ms.to_scipy(mtl5.generators.laplacian_2d(m - 1, m - 1))
zeros = int((K_int.data == 0).sum())
print(f"interior: {K_int.shape[0]} unknowns, {K_int.nnz:,} stored, {zeros:,} of them exactly zero")
print(f"max |K_int - laplacian_2d| = {abs(K_int - L2).max():.1e}")

assert abs(K_int - L2).max() < 1e-14
assert K_int.nnz - zeros == L2.nnz

## Reordering: bandwidth and profile

Now take the Dirichlet system in the mesh generator's random numbering: the
interior unknowns, in label order. Its nonzeros are scattered over the whole
matrix, with a bandwidth close to n. RCM renumbers the unknowns by a
breadth-first sweep from a node at the edge of the mesh, then reverses the
order. Neighbours get nearby numbers, and the nonzeros gather in a band about
as wide as one row of the grid.

The C++ example shows this idea with a hand-picked permutation of a 6 × 6
matrix, which reduces its bandwidth from 5 to 3. RCM does better on the same
matrix: 2.

In [ ]:
A_cpp = np.array(
    [
        [4, 0, 0, -1, 0, -1],
        [0, 4, -1, 0, -1, 0],
        [0, -1, 4, -1, 0, 0],
        [-1, 0, -1, 4, 0, 0],
        [0, -1, 0, 0, 4, -1],
        [-1, 0, 0, 0, -1, 4],
    ],
    dtype=float,
)
hand = [0, 3, 2, 5, 4, 1]
p6 = ms.rcm(ms.from_scipy(sp.csr_matrix(A_cpp)))
bw_cpp = [bandwidth(A_cpp[np.ix_(q, q)]) for q in (range(6), hand, p6)]
print(f"C++ example: bandwidth {bw_cpp[0]}, by hand {bw_cpp[1]}, by RCM {bw_cpp[2]} with p = {p6}")
assert bw_cpp == [5, 3, 2]

keep = np.sort(label[interior])  # the interior unknowns, in label order
A = K[keep][:, keep].tocsr()
n = A.shape[0]
p = ms.rcm(ms.from_scipy(A))  # p[new] = old
A_rcm = A[p][:, p]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), dpi=150)
spy(axes[0], K_int, f"grid order\nbandwidth {bandwidth(K_int)}", color="#8a8984")
spy(axes[1], A, f"random numbering\nbandwidth {bandwidth(A)}", color=SERIES[1])
spy(axes[2], A_rcm, f"RCM\nbandwidth {bandwidth(A_rcm)}", color=SERIES[0])
plt.tight_layout()
plt.show()

print(f"{'numbering':>16} {'bandwidth':>9} {'profile':>8}")
for name, M in (("grid order", K_int), ("random", A), ("RCM", A_rcm)):
    print(f"{name:>16} {bandwidth(M):9} {profile(M):8,}")

assert bandwidth(A) > 0.9 * n
assert bandwidth(A_rcm) <= m  # one grid row, like the grid order's m - 1
assert profile(A_rcm) < profile(K_int) < profile(A) / 5

## A smaller band isn't the least fill

The C++ program's takeaway is "good numbering reduces bandwidth => less fill-in
=> faster solves". The first step is right, and RCM's band is what a banded or
envelope solver wants. Fill in a general sparse factorization depends on more
than the band, though. Inside the band, Cholesky fills in almost every entry,
which is n × bandwidth in all. Minimum degree orderings (AMD), and nested
dissection, give up the band and fill in less.

`mtl5.sparse.cholesky` takes the ordering as a parameter. Its `rcm` is the same
permutation as `ms.rcm`.

In [ ]:
A_m = ms.from_scipy(A)
fill = {}
for o in ("natural", "rcm", "nd", "amd"):
    ch = ms.cholesky(A_m, ordering=o)
    fill[o] = ch.nnz
    bw_o = bandwidth(A[ch.perm][:, ch.perm])
    print(f"{o:>8}: nnz(L) = {ch.nnz:6,}   bandwidth of the ordered matrix {bw_o:4}")
assert np.array_equal(ms.cholesky(A_m, ordering="rcm").perm, p)

fig, ax = plt.subplots(figsize=(6.4, 3.0))
names = list(fill)
labels = ["natural\n(random numbering)", "rcm", "nd", "amd"]
ax.barh(labels, [fill[o] for o in names], color=[SERIES[1], SERIES[0], SERIES[3], SERIES[2]])
ax.set_xlabel("nonzeros in the Cholesky factor L")
ax.set_title(f"Fill of the {n}-unknown P1 system")
ax.invert_yaxis()
plt.show()

# RCM more than halves the fill of the random numbering, and AMD beats RCM.
assert fill["rcm"] < fill["natural"] / 2
assert fill["amd"] < fill["nd"] < fill["rcm"]

## What reordering leaves alone

A symmetric reordering B = PAPᵀ only renames the unknowns, so it is a similarity
transform by an orthogonal matrix. The C++ example checks that it keeps the
trace and the Frobenius norm. It keeps the whole spectrum, and the solution as
well: solving the reordered system and undoing the permutation gives the same
x.

In [ ]:
Ad, Bd = A.toarray(), A_rcm.toarray()
ev_a, ev_b = np.sort(mtl5.eigvalsh(mtl5.matrix(Ad))), np.sort(mtl5.eigvalsh(mtl5.matrix(Bd)))
print(f"trace {np.trace(Ad):.6f} -> {np.trace(Bd):.6f}")
print(f"||.||_F {np.linalg.norm(Ad):.6f} -> {np.linalg.norm(Bd):.6f}")
print(f"max eigenvalue difference {np.abs(ev_a - ev_b).max():.1e}")

assert np.trace(Ad) == np.trace(Bd)  # the same diagonal, permuted
assert abs(np.linalg.norm(Ad) - np.linalg.norm(Bd)) < 1e-12  # summed in another order
assert np.abs(ev_a - ev_b).max() < 1e-11  # eigenvalues up to 8, to a few ulps

f = np.ones(n) / (m * m)  # f = 1, times the area of a node's patch
x = ms.cholesky(A_m, ordering="natural").solve(f).to_numpy()
y = ms.cholesky(ms.from_scipy(A_rcm.tocsr()), ordering="natural").solve(f[p]).to_numpy()
x_back = np.empty(n)
x_back[p] = y  # y[new] is unknown p[new]
print(f"max |x - P^T y| = {np.abs(x - x_back).max():.1e}")
assert np.abs(x - x_back).max() < 1e-13

## Rows, columns, or both

A permutation can act on rows only, on columns only, or on both. With
p[new] = old, as `ms.rcm` returns it, the three are NumPy indexing: `M[p]`
reorders rows, `M[:, p]` columns, and `M[p][:, p]` both. A permutation matrix
P with P[i, p[i]] = 1 gives the same as a product (see
[Views](views.ipynb)).

In [ ]:
M = np.arange(1.0, 10.0).reshape(3, 3)
q = [2, 0, 1]
P = np.eye(3)[q]
print("M[q] =\n", M[q])
print("M[:, q] =\n", M[:, q])
print("M[q][:, q] =\n", M[q][:, q])

assert np.array_equal(P @ M, M[q])
assert np.array_equal(M @ P.T, M[:, q])
assert np.array_equal(P @ M @ P.T, M[q][:, q])

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `shifted_inserter<inserter<compressed2D<double>, update_plus<double>>> ins(K, 4, 0, 0)` | triplets with global indices: `ms.coo_matrix(row, col, data, shape)` |
| `ins.set_row_offset(e); ins[0][0] << v` | `rows += ...`: an element's global indices are its nodes' labels |
| duplicates added by `update_plus` | duplicates summed by `.tocsr()` |
| a hand-picked "Cuthill–McKee-like" permutation | `ms.rcm(A)`, or `ms.ordering(A, "rcm")` |
| `mat::reorder(A, perm)` | `A[perm][:, perm]` (SciPy or NumPy) |
| `mat::reorder_rows(M, p)`, `mat::reorder_cols(M, p)` | `M[p]`, `M[:, p]` |
| `permutation_matrix<double> P(p); P * M` | `np.eye(n)[p] @ M` |

Three changes from the C++ program. The 2-D part assembles a real P1 stiffness
matrix and checks it against `generators.laplacian_2d`, rather than adding a
made-up local matrix. The permutation comes from RCM rather than by hand.
And the fill of the factorization is measured, which shows that the smallest
band isn't the least fill.